# Packet P-038 — Family-Conditional Feature Importance

**Decision question:** Are the bootstrap-stable features (bandgap, thickness, area, anneal_time) equally important across all families, or does importance shift? Does this explain Pure FA's near-zero tau-b?

**Fastest falsifier:** Compare top-3 features for Pure MA vs Triple cation. If identical, family-conditional importance adds nothing.

**Success:** Spearman correlation of importance vectors between Pure MA and Triple cation < 0.7.
**Kill:** All pairwise correlations > 0.9 — features matter equally everywhere.

In [1]:
"""Cell 1 — Load data, train model, compute per-family permutation importance."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)

# Train on 80%, test on 20%
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = ExtraTreesRegressor(**ET_PARAMS)
model.fit(X_tr, y_tr)
print(f"Model trained. Test set: {len(X_te)} devices")

# Compute importance per family on test set
test_families = df.loc[X_te.index, "family"]
importance_by_family = {}

print(f"\n{'=' * 70}")
print("FAMILY-CONDITIONAL PERMUTATION IMPORTANCE")
print(f"{'=' * 70}")

for fam in sorted(test_families.unique()):
    mask = test_families == fam
    n_fam = mask.sum()
    if n_fam < 15:
        print(f"\n{fam} (n={n_fam}): too few devices, skipping")
        continue
    
    X_fam = X_te[mask]
    y_fam = y_te[mask]
    
    perm = permutation_importance(model, X_fam, y_fam, n_repeats=10,
                                   random_state=42, scoring='r2')
    imp = perm.importances_mean
    importance_by_family[fam] = imp
    
    ranked = np.argsort(imp)[::-1]
    print(f"\n{fam} (n={n_fam}) — top-5:")
    for i in range(min(5, len(FEATURES))):
        idx = ranked[i]
        print(f"  {i+1}. {FEATURES[idx]:<40} {imp[idx]:.4f}")

# Feature variance within each family (to explain importance differences)
print(f"\n{'=' * 70}")
print("FEATURE VARIANCE BY FAMILY (std of each feature)")
print(f"{'=' * 70}")
jv_features = ['JV_default_Jsc', 'JV_default_Voc', 'JV_default_FF']
for fam in sorted(df["family"].unique()):
    mask = df["family"] == fam
    n = mask.sum()
    if n < 30:
        continue
    stds = X[mask][jv_features].std()
    print(f"\n{fam} (n={n}):")
    for f in jv_features:
        print(f"  {f}: std={stds[f]:.4f}")

Model trained. Test set: 309 devices

FAMILY-CONDITIONAL PERMUTATION IMPORTANCE

FA-Cs (n=8): too few devices, skipping



MA-FA mixed (n=43) — top-5:
  1. Perovskite_band_gap                      0.1222
  2. Perovskite_thickness                     0.0749
  3. Cs                                       0.0000
  4. Cl                                       0.0000
  5. Sn                                       0.0000



Other (n=22) — top-5:
  1. first_Prvskt_thermal_annealing_time      0.2013
  2. Br                                       0.1994
  3. JV_default_Voc                           0.0905
  4. Sn                                       0.0580
  5. I                                        0.0464

Pure FA (n=6): too few devices, skipping



Pure MA (n=193) — top-5:
  1. Cell_area_measured                       0.1307
  2. Perovskite_band_gap                      0.0749
  3. Perovskite_thickness                     0.0665
  4. JV_default_Jsc                           0.0407
  5. first_Prvskt_annealing_temperature       0.0214



Triple cation (n=37) — top-5:
  1. first_Prvskt_annealing_temperature       0.0300
  2. MA                                       0.0100
  3. Cell_area_measured                       0.0091
  4. Pb                                       0.0074
  5. FA                                       0.0013

FEATURE VARIANCE BY FAMILY (std of each feature)

FA-Cs (n=44):
  JV_default_Jsc: std=2.3638
  JV_default_Voc: std=0.1112
  JV_default_FF: std=0.0621

MA-FA mixed (n=197):
  JV_default_Jsc: std=6.4797
  JV_default_Voc: std=0.3176
  JV_default_FF: std=0.1992

Other (n=88):
  JV_default_Jsc: std=5.2865
  JV_default_Voc: std=0.3126
  JV_default_FF: std=0.1792

Pure FA (n=50):
  JV_default_Jsc: std=4.2152
  JV_default_Voc: std=0.2914
  JV_default_FF: std=0.1016

Pure MA (n=967):
  JV_default_Jsc: std=4.8162
  JV_default_Voc: std=0.2113
  JV_default_FF: std=0.1887

Triple cation (n=197):
  JV_default_Jsc: std=5.0229
  JV_default_Voc: std=0.2483
  JV_default_FF: std=0.1652


In [2]:
"""Cell 2 — Pairwise Spearman correlations and evaluation."""

# Compute pairwise Spearman of importance vectors
fam_names = sorted(importance_by_family.keys())
print(f"\n{'=' * 60}")
print("PAIRWISE SPEARMAN OF IMPORTANCE VECTORS")
print(f"{'=' * 60}")
print(f"{'':>20}", end="")
for f in fam_names:
    print(f"{f[:12]:>14}", end="")
print()

correlations = []
for f1 in fam_names:
    print(f"{f1:>20}", end="")
    for f2 in fam_names:
        rho, _ = spearmanr(importance_by_family[f1], importance_by_family[f2])
        correlations.append({'family1': f1, 'family2': f2, 'spearman': rho})
        print(f"{rho:>14.3f}", end="")
    print()

# Check MA vs Triple cation specifically
if 'Pure MA' in importance_by_family and 'Triple cation' in importance_by_family:
    rho_key, _ = spearmanr(importance_by_family['Pure MA'], importance_by_family['Triple cation'])
    print(f"\nKey comparison — Pure MA vs Triple cation: rho = {rho_key:.3f}")

# All pairwise correlations (excluding self)
cross_corrs = [c['spearman'] for c in correlations if c['family1'] != c['family2']]
mean_cross = np.mean(cross_corrs)
min_cross = np.min(cross_corrs)
print(f"Mean pairwise correlation: {mean_cross:.3f}")
print(f"Min pairwise correlation: {min_cross:.3f}")

# Status
all_above_09 = all(c >= 0.9 for c in cross_corrs)
any_below_07 = any(c < 0.7 for c in cross_corrs)

if all_above_09:
    status = "Negative"  # features matter equally
elif any_below_07:
    status = "Confirmed"  # importance genuinely differs
else:
    status = "Promising"

# Save
corr_df = pd.DataFrame(correlations)
imp_df = pd.DataFrame({fam: importance_by_family[fam] for fam in fam_names}, index=FEATURES)
imp_df.to_csv('Packet_P038_Family_Importance.csv')
print(f"\nSaved: Packet_P038_Family_Importance.csv")

print(f"\n{'=' * 60}")
print(f"P-038 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Feature importance differs significantly across families.")
    print("The 'universal' importance narrative is misleading.")
elif status == "Promising":
    print("Some importance variation but not dramatic.")
else:
    print("Features matter equally across families — universal importance holds.")


PAIRWISE SPEARMAN OF IMPORTANCE VECTORS
                       MA-FA mixed         Other       Pure MA  Triple catio
         MA-FA mixed         1.000        -0.393        -0.222         0.106
               Other        -0.393         1.000        -0.204        -0.340
             Pure MA        -0.222        -0.204         1.000        -0.283
       Triple cation         0.106        -0.340        -0.283         1.000

Key comparison — Pure MA vs Triple cation: rho = -0.283
Mean pairwise correlation: -0.223
Min pairwise correlation: -0.393

Saved: Packet_P038_Family_Importance.csv

P-038 STATUS: Confirmed
Feature importance differs significantly across families.
The 'universal' importance narrative is misleading.
